# How variable is the temperature? Daily, day-over-day and yearly swings in 40 cities, 1980-2025

**Questions**
1. How much does temperature swing *within a day* (diurnal range)?
2. How much does it jump *from one day to the next*?
3. How big is the *yearly* swing from coldest to hottest weeks?
4. Have these changed since 1980?

**Data:** [Open-Meteo Historical API](https://open-meteo.com/en/docs/historical-weather-api), ERA5 reanalysis (~25 km grid, hourly). Same model for every city, so comparisons are not distorted by different instruments or station gaps.

**This notebook so far:** 1) config, 2) cached fetch, 3) hourly to daily table, 4) sanity check, 5) metrics, 6) city summary table. Visuals, trends and validation come next.

## 1. Config

In [ ]:
from pathlib import Path
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

START_YEAR, END_YEAR = 1980, 2025          # full years only
CHUNK_YEARS = 5                            # years per API request
HOURLY_VARS = ["temperature_2m", "apparent_temperature"]   # add "relative_humidity_2m" if you want it
API_URL = "https://archive-api.open-meteo.com/v1/archive"

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
DAILY_DIR = DATA_DIR / "daily"
for d in (RAW_DIR, DAILY_DIR):
    d.mkdir(parents=True, exist_ok=True)

### City list
`tz` is the IANA timezone, used to convert UTC hours to *local* time (so "hour of day" is right and the daily max/min fall on the local calendar day).
`group` is a rough climate type, used later to colour and facet charts.

In [ ]:
cities = [
    # name, lat, lon, tz, group
    ("Mumbai", 19.076, 72.878, "Asia/Kolkata", "Tropical"),
    ("Singapore", 1.352, 103.820, "Asia/Singapore", "Tropical"),
    ("Lagos", 6.524, 3.379, "Africa/Lagos", "Tropical"),
    ("Kinshasa", -4.442, 15.266, "Africa/Kinshasa", "Tropical"),
    ("Dhaka", 23.810, 90.412, "Asia/Dhaka", "Tropical"),
    ("Kolkata", 22.573, 88.364, "Asia/Kolkata", "Tropical"),
    ("Manila", 14.600, 120.984, "Asia/Manila", "Tropical"),
    ("Jakarta", -6.209, 106.846, "Asia/Jakarta", "Tropical"),
    ("Rio de Janeiro", -22.907, -43.173, "America/Sao_Paulo", "Tropical"),

    ("Nairobi", -1.286, 36.817, "Africa/Nairobi", "Highland"),
    ("Mexico City", 19.433, -99.133, "America/Mexico_City", "Highland"),
    ("Bangalore", 12.972, 77.594, "Asia/Kolkata", "Highland"),

    ("Dubai", 25.205, 55.271, "Asia/Dubai", "Arid"),
    ("Phoenix", 33.449, -112.074, "America/Phoenix", "Arid"),
    ("Cairo", 30.044, 31.236, "Africa/Cairo", "Arid"),
    ("Karachi", 24.861, 67.010, "Asia/Karachi", "Arid"),

    ("Delhi", 28.614, 77.209, "Asia/Kolkata", "Humid subtropical"),
    ("Tokyo", 35.690, 139.692, "Asia/Tokyo", "Humid subtropical"),
    ("Osaka", 34.694, 135.502, "Asia/Tokyo", "Humid subtropical"),
    ("Shanghai", 31.230, 121.474, "Asia/Shanghai", "Humid subtropical"),
    ("Guangzhou", 23.129, 113.264, "Asia/Shanghai", "Humid subtropical"),
    ("Chongqing", 29.563, 106.552, "Asia/Shanghai", "Humid subtropical"),
    ("Sao Paulo", -23.551, -46.633, "America/Sao_Paulo", "Humid subtropical"),
    ("Buenos Aires", -34.604, -58.382, "America/Argentina/Buenos_Aires", "Humid subtropical"),
    ("Sydney", -33.869, 151.209, "Australia/Sydney", "Humid subtropical"),
    ("Istanbul", 41.008, 28.978, "Europe/Istanbul", "Humid subtropical"),
    ("Dallas", 32.777, -96.797, "America/Chicago", "Humid subtropical"),

    ("Seattle", 47.606, -122.332, "America/Los_Angeles", "Maritime temperate"),
    ("London", 51.507, -0.128, "Europe/London", "Maritime temperate"),
    ("Los Angeles", 34.052, -118.244, "America/Los_Angeles", "Maritime temperate"),
    ("Cape Town", -33.925, 18.424, "Africa/Johannesburg", "Maritime temperate"),

    ("New York", 40.713, -74.006, "America/New_York", "Continental"),
    ("Toronto", 43.653, -79.383, "America/Toronto", "Continental"),
    ("Chicago", 41.878, -87.630, "America/Chicago", "Continental"),
    ("Minneapolis", 44.978, -93.265, "America/Chicago", "Continental"),
    ("Denver", 39.739, -104.990, "America/Denver", "Continental"),
    ("Moscow", 55.756, 37.617, "Europe/Moscow", "Continental"),
    ("Beijing", 39.904, 116.407, "Asia/Shanghai", "Continental"),
    ("Seoul", 37.567, 126.978, "Asia/Seoul", "Continental"),

    ("Reykjavik", 64.147, -21.942, "Atlantic/Reykjavik", "Subpolar"),
]
cities = pd.DataFrame(cities, columns=["city", "lat", "lon", "tz", "group"])
cities["slug"] = cities["city"].str.lower().str.replace(" ", "_")
print(len(cities), "cities")
cities.groupby("group")["city"].apply(", ".join)

## 2. Fetch (cached and resumable)

- One request per city per 5-year chunk, saved to `data/raw/<city>_<start>_<end>.parquet`. Existing files are skipped, so you can re-run the cell any time.
- Times are requested in **UTC** and converted to local time in step 3. That avoids duplicated or missing hours around daylight-saving changes.
- The free API has per-minute, per-hour and per-day limits, and bigger requests count for more. If the daily limit is hit, the loop stops cleanly. Just re-run the next day and it carries on where it left off. Check Open-Meteo's docs for current limits.
- `data/raw/_grid_info.csv` records the grid point and elevation ERA5 actually used for each city, which is worth a glance for coastal cities like Mumbai.

In [ ]:
class DailyLimitReached(Exception):
    pass


def year_chunks(start_year, end_year, size):
    y = start_year
    while y <= end_year:
        yield y, min(y + size - 1, end_year)
        y += size


def fetch_chunk(row, y0, y1, max_retries=6):
    params = {
        "latitude": row.lat,
        "longitude": row.lon,
        "start_date": f"{y0}-01-01",
        "end_date": f"{y1}-12-31",
        "hourly": ",".join(HOURLY_VARS),
        "timezone": "GMT",
        "models": "era5",
    }
    for attempt in range(max_retries):
        r = requests.get(API_URL, params=params, timeout=120)
        if r.status_code == 200:
            return r.json()
        reason = ""
        try:
            reason = r.json().get("reason", "")
        except Exception:
            pass
        if r.status_code == 429:
            if "daily" in reason.lower():
                raise DailyLimitReached(reason)
            wait = 65 * (attempt + 1)          # minute/hour limit: wait and retry
            print(f"  rate limited ({reason or '429'}), sleeping {wait}s")
            time.sleep(wait)
            continue
        raise RuntimeError(f"HTTP {r.status_code}: {reason or r.text[:200]}")
    raise RuntimeError("too many retries")


def fetch_all(cities_df, pause=0.5):
    grid_rows = []
    for row in cities_df.itertuples():
        for y0, y1 in year_chunks(START_YEAR, END_YEAR, CHUNK_YEARS):
            path = RAW_DIR / f"{row.slug}_{y0}_{y1}.parquet"
            if path.exists():
                continue
            print(f"{row.city} {y0}-{y1}")
            j = fetch_chunk(row, y0, y1)
            df = pd.DataFrame(j["hourly"])
            df["time"] = pd.to_datetime(df["time"], utc=True)
            df.to_parquet(path, index=False)
            grid_rows.append({"city": row.city, "req_lat": row.lat, "req_lon": row.lon,
                              "grid_lat": j.get("latitude"), "grid_lon": j.get("longitude"),
                              "elevation_m": j.get("elevation")})
            time.sleep(pause)
    if grid_rows:
        info_path = RAW_DIR / "_grid_info.csv"
        new = pd.DataFrame(grid_rows).drop_duplicates("city")
        if info_path.exists():
            old = pd.read_csv(info_path)
            new = pd.concat([old, new]).drop_duplicates("city", keep="first")
        new.to_csv(info_path, index=False)


try:
    fetch_all(cities)
    print("Fetch complete.")
except DailyLimitReached as e:
    print("Daily API limit reached. Re-run this cell later to resume.\n", e)

## 3. Hourly to daily table (local time)

For each city: stitch chunks together, convert UTC to local time, then summarise each local calendar day.
Days with fewer than 23 hourly values (the first and last day can be partial after the timezone shift) are dropped.
DST days have 23 or 25 hours, which is why the threshold is 23 rather than 24.

In [ ]:
def build_daily(row):
    files = sorted(RAW_DIR.glob(f"{row.slug}_*.parquet"))
    if not files:
        return None
    h = pd.concat([pd.read_parquet(f) for f in files]).drop_duplicates("time").sort_values("time")
    h["local"] = h["time"].dt.tz_convert(row.tz)
    h["date"] = h["local"].dt.tz_localize(None).dt.normalize()

    g = h.groupby("date")
    daily = pd.DataFrame({
        "n_hours": g["temperature_2m"].count(),
        "tmean": g["temperature_2m"].mean(),
        "tmax": g["temperature_2m"].max(),
        "tmin": g["temperature_2m"].min(),
        "tstd_hourly": g["temperature_2m"].std(),
    })
    if "apparent_temperature" in h:
        daily["app_mean"] = g["apparent_temperature"].mean()
        daily["app_max"] = g["apparent_temperature"].max()
        daily["app_min"] = g["apparent_temperature"].min()
    daily = daily[daily["n_hours"] >= 23].copy()
    daily["dtr"] = daily["tmax"] - daily["tmin"]          # diurnal temperature range
    daily.insert(0, "city", row.city)
    return daily.reset_index()


frames = [build_daily(r) for r in cities.itertuples()]
daily = pd.concat([f for f in frames if f is not None], ignore_index=True)
daily = daily.merge(cities[["city", "group"]], on="city")
daily["year"] = daily["date"].dt.year
daily["month"] = daily["date"].dt.month
daily.to_parquet(DAILY_DIR / "all_cities_daily.parquet", index=False)
print(daily.shape)
daily.head()

## 4. Sanity check

Before computing anything fancy, check coverage and that the quick results match intuition
(Mumbai and Singapore low diurnal range; Phoenix, Denver and Dubai high; Seattle somewhere in between).

In [ ]:
expected_days = (pd.Timestamp(f"{END_YEAR}-12-31") - pd.Timestamp(f"{START_YEAR}-01-01")).days + 1
cov = daily.groupby("city").size().rename("days").to_frame()
cov["pct_complete"] = (cov["days"] / expected_days * 100).round(2)
print("Cities with <99.9% coverage:")
print(cov[cov["pct_complete"] < 99.9])

mean_dtr = (daily.groupby(["group", "city"])["dtr"].mean()
            .reset_index().sort_values("dtr"))

fig, ax = plt.subplots(figsize=(8, 10))
palette = dict(zip(sorted(cities["group"].unique()), plt.cm.tab10.colors))
ax.barh(mean_dtr["city"], mean_dtr["dtr"], color=[palette[g] for g in mean_dtr["group"]])
ax.set_xlabel("Mean diurnal temperature range (deg C), 1980-2025")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in palette.values()]
ax.legend(handles, palette.keys(), loc="lower right", fontsize=8)
ax.set_title("Sanity check: average daily range by city")
plt.tight_layout()
plt.show()

## 5. Metrics

Three scales, three metrics (plus robust and "tail" variants):

| Scale | Primary | Variants |
|---|---|---|
| Within a day | Diurnal range = daily max - daily min | monthly profile |
| Day over day | Mean absolute change in daily mean temp, abs(T(d) - T(d-1)) | same on **anomalies** (seasonal cycle removed); 95th percentile; days per year with a >=10 C change over 2 days |
| Within a year | P95 - P5 of daily mean temp | std of daily means; extreme range (hottest hour - coldest hour); "feels-like" extreme range from apparent temperature |

**Anomaly** = daily mean minus that calendar day's long-term average (smoothed with a 31-day circular window). It strips out the normal spring warm-up and autumn cool-down so the day-over-day metric reflects weather swings, not the seasons.

In [ ]:
d = daily.sort_values(["city", "date"]).reset_index(drop=True)

# day-of-year with leap days folded (Feb 29 and Mar 1 share a slot; Dec 31 folded into 365)
doy = d["date"].dt.dayofyear - ((d["date"].dt.is_leap_year) & (d["date"].dt.month > 2)).astype(int)
d["doy"] = doy.clip(1, 365)

# smoothed day-of-year climatology per city (31-day circular window)
clim = d.groupby(["city", "doy"])["tmean"].mean().unstack("city")
pad = 15
ext = pd.concat([clim.iloc[-pad:], clim, clim.iloc[:pad]])
smooth = ext.rolling(31, center=True).mean().iloc[pad:-pad]
smooth.index = clim.index
clim_long = smooth.stack().rename("clim").reset_index()
d = d.merge(clim_long, on=["city", "doy"], how="left")
d["tanom"] = d["tmean"] - d["clim"]

# day-over-day and 2-day changes, only across consecutive calendar days
g = d.groupby("city")
gap1 = g["date"].diff().dt.days
gap2 = g["date"].diff(2).dt.days
d["d1_raw"] = g["tmean"].diff().where(gap1 == 1)
d["d1_anom"] = g["tanom"].diff().where(gap1 == 1)
d["d2_raw"] = (d["tmean"] - g["tmean"].shift(2)).where(gap2 == 2)

SWING_C = 10   # threshold (deg C) for a "big swing within 2 days"
d["abs_d1_raw"] = d["d1_raw"].abs()
d["abs_d1_anom"] = d["d1_anom"].abs()
d["swing48"] = (d["d2_raw"].abs() >= SWING_C).astype(int)
d[["city", "date", "tmean", "clim", "tanom", "d1_raw", "d1_anom", "d2_raw"]].tail()

In [ ]:
annual = d.groupby(["city", "year"]).agg(
    dtr_mean=("dtr", "mean"),
    dod_abs_raw=("abs_d1_raw", "mean"),
    dod_abs_anom=("abs_d1_anom", "mean"),
    dod_p95_anom=("abs_d1_anom", lambda s: s.quantile(0.95)),
    swing48_days=("swing48", "sum"),
    tmean_p05=("tmean", lambda s: s.quantile(0.05)),
    tmean_p95=("tmean", lambda s: s.quantile(0.95)),
    annual_std=("tmean", "std"),
    tmax_year=("tmax", "max"),
    tmin_year=("tmin", "min"),
    app_max_year=("app_max", "max"),
    app_min_year=("app_min", "min"),
).reset_index()

annual["annual_p95_p5"] = annual["tmean_p95"] - annual["tmean_p05"]
annual["annual_extreme_range"] = annual["tmax_year"] - annual["tmin_year"]
annual["feels_like_extreme_range"] = annual["app_max_year"] - annual["app_min_year"]
annual = annual.merge(cities[["city", "group"]], on="city")

# monthly diurnal-range profile (city x month), for seasonal contrast later
dtr_month = d.groupby(["city", "month"])["dtr"].mean().unstack("month")

annual.to_parquet(DAILY_DIR / "annual_metrics.parquet", index=False)
d.to_parquet(DAILY_DIR / "daily_with_anomalies.parquet", index=False)
annual.head()

## 6. City summary (1980-2025 averages)

One row per city. Quick check of your hypotheses: Mumbai should have a low diurnal range, Seattle a larger one,
Dallas/New York/Denver a high day-over-day variability and many big-swing days, and Toronto/Moscow/Minneapolis a huge yearly range.

In [ ]:
metrics = ["dtr_mean", "dod_abs_anom", "dod_p95_anom", "swing48_days",
           "annual_p95_p5", "annual_extreme_range", "feels_like_extreme_range"]
summary = annual.groupby("city")[metrics].mean().round(1)
summary = summary.join(cities.set_index("city")["group"])

pd.set_option("display.max_rows", 60)
summary.sort_values("annual_p95_p5", ascending=False)

In [ ]:
# rank (1 = most variable) for the cities named in the hypotheses
ranks = summary[metrics].rank(ascending=False).astype(int)
ranks.loc[["Mumbai", "Seattle", "Dallas", "New York", "Toronto"]]

## Next
7. Visuals: temperature carpets, ECDFs of day-over-day change, variability fingerprint scatter, seasonal bands
8. Trends: Theil-Sen slope per city and metric, 1980s vs 2016-2025
9. Validation against NOAA/Meteostat stations for a few cities